## Feature Documentation (Monthly, Sector ETFs + Risk-Free Asset)

This project constructs a monthly feature panel for **reinforcement learning–based dynamic portfolio allocation** across U.S. sector ETFs, augmented with a **risk-free asset**.  
All features are computed at **month-end**, using only information available up to that point, consistent with a **monthly rebalance** setting.

---

## Dataset Structure (`panel`)
- **Index:** `(date, ticker)`
- **Frequency:** Monthly (Jan 2016 – Jan 2026)
- **Investable Assets:**
  - 11 U.S. sector ETFs (GICS-style coverage)
  - 1 risk-free / cash-like asset

### Asset Universe

| ETF | Sector / Role |
|----|---------------|
| VOX | Communication Services |
| XLY | Consumer Discretionary |
| XLP | Consumer Staples |
| XLE | Energy |
| XLF | Financials |
| XLV | Health Care |
| XLI | Industrials |
| XLK | Information Technology |
| XLB | Materials |
| XLRE | Real Estate |
| XLU | Utilities |
| BIL | Risk-Free Asset (1–3 Month U.S. Treasury Bills |

**Note on BIL:**  
BIL serves as a **cash / risk-free allocation option** for the RL agent, enabling capital preservation, volatility control, and regime-dependent de-risking.

---

## Asset-Level Features

### 1. One-Month Return (`ret_1m`)
Simple one-month return of each ETF.
$$
\text{ret\_1m}_{i,t} = \frac{P_{i,t} - P_{i,t-1}}{P_{i,t-1}}
$$
**Interpretation:**  
Captures recent performance and short-term return dynamics relevant for tactical allocation.

---

### 2. One-Month Realized Volatility (`vol_1m`)
Standard deviation of **daily returns within month $t$**.
$$
\text{vol\_1m}_{i,t} = \sqrt{\text{Var}(r_{i,d})}, \quad d \in t
$$
**Interpretation:**  
Measures *within-asset* risk over the past month.  
Higher values indicate greater short-term uncertainty.

---

### 3. Amihud Illiquidity (`amihud_1m`)
Monthly Amihud (2002) illiquidity proxy computed from daily data.
$$
\text{amihud\_1m}_{i,t}
= \frac{1}{D_t} \sum_{d \in t}
\frac{|r_{i,d}|}{P_{i,d} \times \text{Volume}_{i,d}}
$$
**Interpretation:**  
Proxies price impact and trading costs.  
**Higher values imply worse liquidity and higher execution frictions.**

---

## Market / Cross-Asset Context Features  
*(Identical across all assets within the same month)*

> **Important:**  
> The following two features are computed **using only the 11 sector ETFs**.  
> **BIL is explicitly excluded** to avoid mechanically biasing these measures toward low correlation or low dispersion due to its cash-like behavior.

---

### 4. Average Cross-Sector Correlation (`avg_corr_1m`)
Average pairwise correlation across **sector ETFs** using daily returns within month $t$.
$$
\text{avg\_corr\_1m}_t
= \frac{1}{N(N-1)} \sum_{i \neq j} \rho_{i,j,t}
$$

**How to interpret “high” vs “low” (relative to history):**
- **High correlation:**  
  - Sectors move together  
  - Diversification benefits are reduced  
  - Typical of **stress, crisis, or risk-off regimes**
- **Low correlation:**  
  - Sectors behave more independently  
  - Greater diversification potential  
  - Often observed in **stable or transitioning regimes**

There is no absolute cutoff; interpretation is **relative to the historical distribution** of this feature.

---

### 5. Cross-Sectional Dispersion (`xsec_disp_1m`)
Standard deviation of **sector-level one-month returns**, computed across sector ETFs.
$$
\text{xsec\_disp\_1m}_t
= \sqrt{\text{Var}(r_{1,t}, r_{2,t}, \dots, r_{N,t})}
$$

**How to interpret “high” vs “low” (relative to history):**
- **High dispersion:**  
  - Large performance gaps between sectors  
  - Clear winners and losers  
  - Indicates strong **sector rotation**, heterogeneous economic shocks, or regime shifts  
  - Favors **active allocation and relative-value strategies**
- **Low dispersion:**  
  - Sector returns are tightly clustered  
  - Broad market forces dominate  
  - Fewer gains from active sector tilting

**Key distinction from volatility:**
- **Dispersion** → differences *across sectors* at a point in time  
- **Volatility** → fluctuations *within an asset* over time

---

## How the Market / Cross-Asset Context Features Can Help Decision Making

Together, `avg_corr_1m` and `xsec_disp_1m` describe the **structure of the opportunity set** faced by the RL agent:

- High correlation → limited diversification
- High dispersion → richer allocation opportunities
- Low dispersion → market-wide behavior dominates
- Presence of **BIL** → enables dynamic risk-off behavior

All regime features are interpreted **relative to their own historical distributions**, aligning with standard empirical finance practice and avoiding hard-coded thresholds.


In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

sector_tickers = ["VOX","XLY","XLP","XLE","XLF","XLV","XLI","XLK","XLB","XLRE","XLU"]
tickers = sector_tickers + ["BIL"]

start_date = "2015-12-01"   # start one month earlier
end_date = "2026-01-31"

raw = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=False,
    group_by="column",
    progress=False,
    threads=True
)

adj_close = raw["Adj Close"].copy().dropna(how="all")
volume = raw["Volume"].copy().reindex(adj_close.index)

FREQ = "ME"  # month-end

# Month-end prices and monthly returns
px_m = adj_close.resample(FREQ).last()
ret_1m = px_m.pct_change(fill_method=None) 

# Daily returns
ret_d = adj_close.pct_change()

# 1M realized vol: std of daily returns within month
vol_1m = ret_d.resample(FREQ).std()

# Liquidity: Amihud illiquidity (monthly average of |r_d| / dollar volume) (higher means worse liquidity)
dollar_vol_d = (adj_close * volume).replace(0, np.nan)
amihud_d = ret_d.abs() / dollar_vol_d
amihud_1m = amihud_d.resample(FREQ).mean()

# average pairwise correlation across sectors within each month
def avg_offdiag_corr_df(df_month: pd.DataFrame) -> float:
    df_month = df_month.dropna(axis=1, how="all").dropna(how="any")
    if df_month.shape[1] < 2 or df_month.shape[0] < 2:
        return np.nan
    C = df_month.corr().to_numpy()
    n = C.shape[0]
    return (C.sum() - np.trace(C)) / (n * (n - 1))

# compute correlation using sector ETFs only (exclude BIL)
avg_corr_1m = (
    ret_d[sector_tickers].groupby(pd.Grouper(freq=FREQ))
                         .apply(avg_offdiag_corr_df)
)

# Cross-sectional dispersion of monthly returns
# dispersion across sector ETFs only (exclude BIL)
xsec_disp_1m = ret_1m[sector_tickers].std(axis=1)

def wide_to_long(df_wide, value_name):
    """
    Convert a (date x ticker) wide DataFrame to long format:
    columns: date, ticker, <value_name>
    Keeps NaNs.
    """
    out = (
        df_wide
        .reset_index()
        .melt(id_vars=df_wide.index.name or "index", var_name="ticker", value_name=value_name)
    )
    # rename index column to "date"
    out = out.rename(columns={df_wide.index.name or "index": "date"})
    return out

# build features_asset and broadcast market context to all tickers
monthly_index = px_m.index  # full month-end index including 2015-12-31

avg_corr_df = pd.DataFrame(
    np.repeat(avg_corr_1m.reindex(monthly_index).values.reshape(-1, 1), len(tickers), axis=1),
    index=monthly_index,
    columns=tickers
)

xsec_disp_df = pd.DataFrame(
    np.repeat(xsec_disp_1m.reindex(monthly_index).values.reshape(-1, 1), len(tickers), axis=1),
    index=monthly_index,
    columns=tickers
)

features_asset = {
    "ret_1m": ret_1m.reindex(monthly_index),
    "vol_1m": vol_1m.reindex(monthly_index),
    "amihud_1m": amihud_1m.reindex(monthly_index),
    "avg_corr_1m": avg_corr_df,
    "xsec_disp_1m": xsec_disp_df,
}

# Build long tables for each feature (keeps NaNs)
long_tables = []
for feat_name, df_feat in features_asset.items():
    df_feat = df_feat.copy()
    df_feat.index.name = "date"
    long_tables.append(wide_to_long(df_feat, feat_name))

# Merge all features on (date, ticker)
panel = long_tables[0]
for tbl in long_tables[1:]:
    panel = panel.merge(tbl, on=["date", "ticker"], how="outer")

# Set multiIndex
panel = panel.set_index(["date", "ticker"]).sort_index()

# create clean panel by dropping rows with any NaNs
panel_clean = panel.dropna()

panel_clean = (
    panel_clean
    .reset_index()
    .assign(date=lambda x: x["date"].dt.normalize())  # strips time
    .set_index(["date", "ticker"])
    .sort_index()
)

print("Panel shape (with NaNs):", panel.shape)
print("Clean panel shape (dropna):", panel_clean.shape)

print("Panel date range:", panel.index.get_level_values("date").min(), "to",
      panel.index.get_level_values("date").max())
print("Clean panel date range:", panel_clean.index.get_level_values("date").min(), "to",
      panel_clean.index.get_level_values("date").max())

# which features are causing missingness (counts over entire panel)
print("\nMissing values per feature (full panel):")
print(panel.isna().sum().sort_values(ascending=False))

display(panel_clean)


Panel shape (with NaNs): (1464, 5)
Clean panel shape (dropna): (1452, 5)
Panel date range: 2015-12-31 00:00:00 to 2026-01-31 00:00:00
Clean panel date range: 2016-01-31 00:00:00 to 2026-01-31 00:00:00

Missing values per feature (full panel):
ret_1m          12
xsec_disp_1m    12
vol_1m           0
amihud_1m        0
avg_corr_1m      0
dtype: int64


ret_1m    vol_1m     amihud_1m  avg_corr_1m  xsec_disp_1m
date       ticker                                                             
2016-01-31 BIL     0.000219  0.000225  3.220861e-12     0.750529      0.046963
           VOX     0.012276  0.016287  2.306592e-09     0.750529      0.046963
           XLB    -0.107094  0.017291  4.828052e-11     0.750529      0.046963
           XLE    -0.034980  0.029289  2.081823e-11     0.750529      0.046963
           XLF    -0.088544  0.016266  1.079141e-11     0.750529      0.046963
...                     ...       ...           ...          ...           ...
2026-01-31 XLP     0.075052  0.008910  4.130291e-12     0.244477      0.048466
           XLRE    0.026766  0.008588  1.681946e-11     0.244477      0.048466
           XLU     0.013118  0.009301  6.939042e-12     0.244477      0.048466
           XLV    -0.000388  0.009066  3.312761e-12     0.244477      0.048466
           XLY     0.014739  0.011190  6.939466e-12     0.244477      0.048466

[1452 rows x 5 columns]

In [2]:
missing_rows = panel[panel.isna().any(axis=1)]

print("Number of rows with missing values:", missing_rows.shape[0])
display(missing_rows)

Number of rows with missing values: 12


ret_1m    vol_1m     amihud_1m  avg_corr_1m  xsec_disp_1m
date       ticker                                                           
2015-12-31 BIL        NaN  0.000183  4.709880e-12     0.740138           NaN
           VOX        NaN  0.012172  2.537911e-09     0.740138           NaN
           XLB        NaN  0.015484  5.631789e-11     0.740138           NaN
           XLE        NaN  0.020303  1.702381e-11     0.740138           NaN
           XLF        NaN  0.014560  1.343445e-11     0.740138           NaN
           XLI        NaN  0.011133  1.698982e-11     0.740138           NaN
           XLK        NaN  0.012312  2.214702e-11     0.740138           NaN
           XLP        NaN  0.010775  2.348419e-11     0.740138           NaN
           XLRE       NaN  0.011247  3.806620e-07     0.740138           NaN
           XLU        NaN  0.010922  2.239064e-11     0.740138           NaN
           XLV        NaN  0.011690  1.539238e-11     0.740138           NaN
           XLY        NaN  0.011403  1.812937e-11     0.740138           NaN

In [3]:
panel_clean.to_parquet("sector_panel.parquet")